In [1]:
import os
import pandas as pd
import polars as pl
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import kaggle_evaluation.default_inference_server
import warnings
warnings.filterwarnings('ignore')

# ULTRA-CONSERVATIVE - Narrower multiplier range for stability
BULL_BASE_MULT = 1280.0      # V33: 1260
BEAR_BASE_MULT = 1020.0      # V33: 1040
NEUTRAL_BASE_MULT = 1160.0   # V33: 1155
TARGET_VOLATILITY = 0.0120   # V33: 0.0118 (slightly less aggressive)
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

print("="*70)
print("V48: ULTRA-CONSERVATIVE - Minimal Risk")
print("="*70)

train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def create_features(df):
    """V33's exact proven features"""
    df = df.copy()
    
    key_cols = ['M1', 'M2', 'M3', 'P1', 'P2', 'S1', 'S2', 'I1', 'I2']
    
    for col in key_cols:
        if col in df.columns:
            df[f'{col}_lag1'] = df[col].shift(1)
            df[f'{col}_lag2'] = df[col].shift(2)
            df[f'{col}_lag5'] = df[col].shift(5)
    
    for col in ['M1', 'M2', 'P1', 'S1']:
        if col in df.columns:
            df[f'{col}_roll5_mean'] = df[col].rolling(5).mean()
            df[f'{col}_roll10_mean'] = df[col].rolling(10).mean()
            df[f'{col}_roll20_mean'] = df[col].rolling(20).mean()
            df[f'{col}_roll5_std'] = df[col].rolling(5).std()
            df[f'{col}_roll10_std'] = df[col].rolling(10).std()
    
    if 'M1' in df.columns and 'M2' in df.columns:
        df['momentum_M'] = df['M1'] - df['M2']
        df['M1_M2_ratio'] = df['M1'] / (df['M2'].abs() + 0.001)
    if 'P1' in df.columns and 'P2' in df.columns:
        df['momentum_P'] = df['P1'] - df['P2']
        df['P1_P2_ratio'] = df['P1'] / (df['P2'].abs() + 0.001)
    
    for col in ['M1', 'P1', 'S1']:
        if col in df.columns:
            df[f'{col}_vol10'] = df[col].rolling(10).std()
            df[f'{col}_vol20'] = df[col].rolling(20).std()
    
    if 'M1' in df.columns:
        delta = df['M1'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = -delta.where(delta < 0, 0).rolling(14).mean()
        rs = gain / (loss + 0.001)
        df['M1_rsi'] = 100 - (100 / (1 + rs))
    
    if 'M1' in df.columns and 'P1' in df.columns:
        df['M1_P1_ratio'] = df['M1'] / (df['P1'].abs() + 0.001)
    
    if 'M1' in df.columns:
        ema12 = df['M1'].ewm(span=12, adjust=False).mean()
        ema26 = df['M1'].ewm(span=26, adjust=False).mean()
        df['M1_macd'] = ema12 - ema26
    
    if 'M1' in df.columns:
        df['M1_roc10'] = (df['M1'] - df['M1'].shift(10)) / (df['M1'].shift(10).abs() + 0.001)
    
    if 'P1' in df.columns:
        ma20 = df['P1'].rolling(20).mean()
        std20 = df['P1'].rolling(20).std()
        df['P1_bb_pos'] = (df['P1'] - ma20) / (std20 + 0.001)
    
    if 'M1' in df.columns:
        ema5 = df['M1'].ewm(span=5, adjust=False).mean()
        ema20 = df['M1'].ewm(span=20, adjust=False).mean()
        df['M1_ema_cross'] = (ema5 - ema20) / (ema20.abs() + 0.001)
    
    if 'M1' in df.columns:
        vol5 = df['M1'].rolling(5).std()
        vol20 = df['M1'].rolling(20).std()
        df['M1_vol_regime'] = vol5 / (vol20 + 0.001)
    
    if 'P1' in df.columns:
        mom = df['P1'].diff()
        df['P1_mom_accel'] = mom.diff()
    
    return df

train = create_features(train)
train = train.dropna()

target_col = 'market_forward_excess_returns'
exclude_cols = ['date_id', 'forward_returns', 'risk_free_rate', target_col]
feature_cols = [col for col in train.columns if col not in exclude_cols]

X_train = train[feature_cols]
y_train = train[target_col]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
imputer = SimpleImputer(strategy='median')
X_train_filled = imputer.fit_transform(X_train_scaled)

# Exact V33 config
model_lgb = lgb.LGBMRegressor(
    n_estimators=460, learning_rate=0.03, max_depth=6,
    num_leaves=31, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42,
    verbose=-1, n_jobs=-1
)
model_lgb.fit(X_train_filled, y_train)

model_cat = CatBoostRegressor(
    iterations=460, learning_rate=0.03, depth=6,
    l2_leaf_reg=3, random_state=42, verbose=0
)
model_cat.fit(X_train_filled, y_train)

model_xgb = XGBRegressor(
    n_estimators=460, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1,
    reg_lambda=0.1, random_state=42, n_jobs=-1, verbosity=0
)
model_xgb.fit(X_train_filled, y_train)

print("✓ Models ready!")

recent_returns_history = train[target_col].tail(60).values

def detect_regime(returns_history):
    """V33's proven regime detection"""
    recent_returns = returns_history[-30:]
    
    mean_return = np.mean(recent_returns)
    short_ma = np.mean(returns_history[-10:])
    long_ma = np.mean(returns_history[-30:])
    
    if mean_return > 0.0005 and short_ma > long_ma:
        return 'bull'
    elif mean_return < -0.0005 and short_ma < long_ma:
        return 'bear'
    else:
        return 'neutral'

def predict(test: pl.DataFrame) -> float:
    """Ultra-conservative with tight multiplier range"""
    global recent_returns_history
    
    test_pd = test.to_pandas()
    test_pd = create_features(test_pd)
    
    for feat in feature_cols:
        if feat not in test_pd.columns:
            test_pd[feat] = 0
    
    X_test = test_pd[feature_cols].iloc[-1:]
    X_test_scaled = scaler.transform(X_test)
    X_test_filled = imputer.transform(X_test_scaled)
    
    pred_lgb = model_lgb.predict(X_test_filled)[0]
    pred_cat = model_cat.predict(X_test_filled)[0]
    pred_xgb = model_xgb.predict(X_test_filled)[0]
    
    raw_prediction = (pred_lgb + pred_cat + pred_xgb) / 3.0
    
    regime = detect_regime(recent_returns_history)
    
    if regime == 'bull':
        base_mult = BULL_BASE_MULT
    elif regime == 'bear':
        base_mult = BEAR_BASE_MULT
    else:
        base_mult = NEUTRAL_BASE_MULT
    
    recent_vol = np.std(recent_returns_history[-20:]) if len(recent_returns_history) >= 20 else np.std(recent_returns_history)
    
    vol_scalar = TARGET_VOLATILITY / (recent_vol + 0.001)
    vol_scalar = np.clip(vol_scalar, 0.5, 1.5)
    
    adaptive_multiplier = base_mult * vol_scalar
    
    signal = raw_prediction * adaptive_multiplier + 1.0
    signal = np.clip(signal, MIN_SIGNAL, MAX_SIGNAL)
    
    return float(signal)

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

print("V48 READY - ULTRA-CONSERVATIVE!")
print("Expected: 8.15 - 8.25 (Very Safe)")

V48: ULTRA-CONSERVATIVE - Minimal Risk
✓ Models ready!
V48 READY - ULTRA-CONSERVATIVE!
Expected: 8.15 - 8.25 (Very Safe)
